In [ ]:
import os
import time

from dotenv import load_dotenv
from pinecone_setup import Pinecone, ServerlessSpec

load_dotenv()

api_key = os.getenv("PINECONE_API_KEY")

if not api_key:
    raise ValueError("PINECONE_API_KEY is not set")

pc = Pinecone(api_key=api_key)

INDEX_NAME = "contextflow"

# Gemini gemini-embedding-001 produces 3072-dimensional
# embeddings by default.
DIMENSION = 3072

if not pc.has_index(INDEX_NAME):
    print(f"Creating Pinecone index: {INDEX_NAME}")

    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        ),
    )

    # Give Pinecone a moment to initialize the index.
    time.sleep(5)

else:
    print(f"Index already exists: {INDEX_NAME}")

index = pc.Index(INDEX_NAME)

print("Pinecone index is ready!")
print(index.describe_index_stats())

Creating Pinecone index: contextflow
Pinecone index is ready!
DescribeIndexStatsResponse(dimension=3072, total_vector_count=0, metric='cosine', namespaces=0)


In [8]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from pinecone import Pinecone

from chunking import chunk_text
from ingestion import load_document


# --------------------------------------------------
# Load environment variables
# --------------------------------------------------
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")
gemini_api_key = os.getenv("GEMINI_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")

if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY is not set")

if not pinecone_api_key:
    raise ValueError("PINECONE_API_KEY is not set")


# --------------------------------------------------
# Gemini embeddings
# --------------------------------------------------

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=gemini_api_key,
)


# --------------------------------------------------
# Pinecone
# --------------------------------------------------

pc = Pinecone(api_key=pinecone_api_key)

INDEX_NAME = "contextflow"

index = pc.Index(INDEX_NAME)


# --------------------------------------------------
# Load and chunk document
# --------------------------------------------------

document_path = PROJECT_ROOT / "data" / "sample.txt"

text = load_document(str(document_path))

chunks = chunk_text(text)

print(f"Loaded document: {document_path.name}")
print(f"Number of chunks: {len(chunks)}")


# --------------------------------------------------
# Create embeddings
# --------------------------------------------------

print("Creating embeddings...")

vectors = embeddings.embed_documents(chunks)

print(f"Created {len(vectors)} embeddings")


# --------------------------------------------------
# Prepare Pinecone records
# --------------------------------------------------

records = []

for i, (chunk, vector) in enumerate(zip(chunks, vectors)):

    record = {
        "id": f"sample-{i}",
        "values": vector,
        "metadata": {
            "text": chunk,
            "source": "sample.txt",
            "chunk_index": i,
        },
    }

    records.append(record)


# --------------------------------------------------
# Upload to Pinecone
# --------------------------------------------------

print("Uploading vectors to Pinecone...")

index.upsert(vectors=records)

print("Upload complete!")


# --------------------------------------------------
# Show Pinecone statistics
# --------------------------------------------------

stats = index.describe_index_stats()

print("\nPinecone index statistics:")
print(stats)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Loaded document: sample.txt
Number of chunks: 1
Creating embeddings...


GoogleGenerativeAIError: Error embedding content (PERMISSION_DENIED): 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your project has been denied access. Please contact support.', 'status': 'PERMISSION_DENIED'}}

In [9]:
from dotenv import load_dotenv

load_dotenv()

import os

gemini_api_key = os.getenv("GEMINI_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")

print("Gemini key loaded:", bool(gemini_api_key))
print("Pinecone key loaded:", bool(pinecone_api_key))

Gemini key loaded: True
Pinecone key loaded: True


In [13]:
import os

from dotenv import load_dotenv
from xai_sdk import Client
from xai_sdk.chat import user

load_dotenv()

api_key = os.getenv("XAI_API_KEY")

if not api_key:
    raise ValueError("XAI_API_KEY is not set")

print("XAI API key loaded:", True)
print("Key length:", len(api_key))

client = Client(api_key=api_key)

chat = client.chat.create(model="grok-4.7")

chat.append(
    user("Say hello in one sentence.")
)

response = chat.sample()

print(response.content)

XAI API key loaded: True
Key length: 56


_InactiveRpcError: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.INVALID_ARGUMENT
	details = "Incorrect API key provided. You can obtain an API key from https://console.x.ai."
	debug_error_string = "INVALID_ARGUMENT:Incorrect API key provided. You can obtain an API key from https://console.x.ai."
>

In [7]:
import os
from dotenv import load_dotenv

load_dotenv()

key = os.getenv("XAI_API_KEY")

print("Key found:", bool(key))
print("Key length:", len(key) if key else 0)
print("Key preview:", key[:6] + "..." if key else "NOT FOUND")

Key found: True
Key length: 84
Key preview: xai-88...


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

key = os.getenv("XAI_API_KEY")

print("Loaded:", bool(key))
print("Prefix:", key[:4] if key else "NONE")
print("Length:", len(key) if key else 0)

Loaded: True
Prefix: xai-
Length: 84


In [2]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

print("Gemini API key loaded:", bool(GOOGLE_API_KEY))

Gemini API key loaded: True


In [3]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=GOOGLE_API_KEY,
)

In [4]:
vector = embeddings.embed_query("What is Retrieval-Augmented Generation?")

print("Embedding worked!")
print("Dimensions:", len(vector))
print("First 5 values:", vector[:5])

Embedding worked!
Dimensions: 3072
First 5 values: [-0.023438238, 0.00724161, -0.00015789657, -0.07040447, -0.012659747]


In [5]:
from pinecone import Pinecone
import os

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

pc = Pinecone(api_key=PINECONE_API_KEY)

index = pc.Index("contextflow")

print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=3072, total_vector_count=0, metric='cosine', namespaces=0)
